# Bank + prompt, best epoch9 — TEST5000

**T4 ×2**, Internet ON, Save & Run All. Attach THREE Datasets:
- `ducanh2403/visual-cache`
- `vuthetam/mscoco-2014`
- `ducanh2403/csa-prompt-test-cache-v181`

Enable your **Kaggle API** Secret `codex` or `KAGGLE_API_TOKEN` (not an OpenAI key). Blank CHECKPOINT_ROOT downloads only best.bin from **ducanh2403/kltnnnnnff/9**, exact SHA7c11be52. User confirmed notebook slug/version; default owner is own ducanh2403. Missing access fails early; no latest/other checkpoint fallback. Alternatively mount the exact knownSHA file/root and set CHECKPOINT_ROOT. No API token printed.

Best epoch9 is159399updates; the full experiment ran10epochs but epoch10 has different weights. Keep epoch9 chosen on VALID before TEST. Model frozen, no training/selection/reranking. Frozen TRAIN noun/verb banks are inside the checkpoint; no bank130 or prompt175 dataset needed. TEST181 only supplies258 fixed prompt embeddings +instance links. Uses exact same ROI-box→14×14patch pooling/masks as training; old ROI feature arrays and old181 vocabulary are NOT used. No CLIP/UPT inference.

Only5000TEST patches staged (~1.4GiB), no full33GiB copy or TRAIN/VALID feature reads. Same5000IDs/rawreferences, TRAIN vocabulary and beam5/max20/rawlogprob/rank0 as baseline199. Captions saved before teacherNLL and CPUscoring. Metrics: BLEU1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall,NLL). Complete only when all eight metrics finished. Previous project TEST exposure disclosed; no TEST tuning.

Parent2h STOP cap inherited from baselineTEST, not ETA/billing. Latest answer confirmed cache path but did not supply remaining GPU hours; check account quota before launch. GPUweights/runtime/firstbeam-path guards before full decode. No Torch reinstall. Preserve caption/output on errors; no automatic model rerun. Send **clip_patch_bank_prompt_test5000_v1_review.zip**. Author checks NumPy/mock/AST only; models/CUDA/scorers run only by user.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-test-cache-v181')
CHECKPOINT_ROOT='' # Optional exact bank+prompt best9 mounted file/root. Blank downloads ONLY own kltnnnnnff/9/best.bin.
SECRET_NAMES=('codex','KAGGLE_API_TOKEN') # Enable the matching Kaggle Secret; token is never logged.
WALL_HOURS=2 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/clip_patch_bank_prompt_test5000_v1')
for p in (VISUAL_CACHE,COCO_JSON,PROMPT_TEST_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct original inputs: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for TEST patches/checkpoint/CPU scorer assets')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=str(OUTPUT),stage=str(SOURCE/'test_patches'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),checkpoint_root=CHECKPOINT_ROOT,chair=str(CHAIR),worker_seconds=1800)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
ROUTING_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output('ducanh2403/kltnnnnnff/9',path=directory)\n    request=endpoint.call_args.args[0]\n    assert (request.user_name,request.kernel_slug,request.version_label)==('ducanh2403','kltnnnnnff','v9')\nprint('Exact own9 API routing mock PASS; no network.',flush=True)\n"
complete=False
try:
    for path,name in MAPPING.items():download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_test199.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_test.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    sys.path.insert(0,str(SOURCE))
    from semantic_test_data import checkpoint,VERSION,CHECKPOINT_SHA,prompt_inputs
    existing=Path('/kaggle/working/clip_patch_bank_prompt_xe10_v1/best.bin')
    available=CHECKPOINT_ROOT or (str(existing) if existing.is_file() else '')
    if available:
        weights=checkpoint(available);origin='mounted or existing exact best9 bytes'
    else:
        run([sys.executable,'-m','pip','install','-q','--target',CLI,'https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],600)
        env=os.environ.copy();env['PYTHONPATH']=str(CLI)
        run([sys.executable,'-u','-c',ROUTING_PREFLIGHT],120,env)
        from kaggle_secrets import UserSecretsClient
        token=None
        for name in SECRET_NAMES:
            try:value=UserSecretsClient().get_secret(name).strip()
            except Exception:continue
            if value:token=value;break
        if not token:raise RuntimeError('Enable own Kaggle API Secret codex or KAGGLE_API_TOKEN; no token is printed.')
        if token.startswith('sk-'):raise RuntimeError('Use a Kaggle API Secret, not an OpenAI key.')
        TOKENS.append(token)
        for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):env.pop(key,None)
        env['KAGGLE_API_TOKEN']=token;env['PYTHONUNBUFFERED']='1'
        target=OUTPUT/'inputs199';target.mkdir()
        # CLI main entry works even on builds without kaggle.__main__.
        run([sys.executable,'-u','-c','from kaggle.cli import main; main()','kernels','output',VERSION,'-p',str(target),'--file-pattern',r'(^|/)best[.]bin$'],900,env)
        weights=checkpoint(target);origin='exact own kltnnnnnff/9 API'
    cfg['checkpoint_root']=str(weights);write_config()
    prompt_inputs(cfg) # exact TEST181 seal/scope check before expensive setup/stage
    (OUTPUT/'checkpoint_source.json').write_text(json.dumps(dict(version=VERSION,sha256=CHECKPOINT_SHA,path=str(weights),origin=origin),indent=2))
    run([sys.executable,'-u',SOURCE/'run_test199.py','checkpoint_preflight','--config',CONFIG],120)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],300)
    base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
    for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d','chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():download(base+name,digest,CHAIR/name)
    run([sys.executable,'-u',SOURCE/'run_test199.py','environment','--config',CONFIG],1200)
    run([sys.executable,'-u',SOURCE/'run_test199.py','smoke','--config',CONFIG],600)
    run([sys.executable,'-u',SOURCE/'run_test199.py','prepare','--config',CONFIG],1800)
    cfg['worker_seconds']=min(1800,PARENT_DEADLINE-time.monotonic()-1800);write_config()
    if cfg['worker_seconds']<300:raise TimeoutError('Insufficient budget before TEST inference; no automatic extension')
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'run_test199.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    run([sys.executable,'-u',SOURCE/'run_test199.py','tokenize','--config',CONFIG],180)
    for mode,cap in (('basic',180),('meteor',600),('spice',1800),('chair',600),('combine',120)):
        run([sys.executable,'-u',SOURCE/'run_test199.py',mode,'--config',CONFIG],cap)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='CLIP_PATCH_BANK_PROMPT_TEST5000_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(redact(traceback.format_exc()));raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_tuning=False,source_version='ducanh2403/kltnnnnnff/9')))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP OUTPUT:',OUTPUT,flush=True)
    print('TEST COMPLETE; no training/tuning.' if complete else 'INCOMPLETE; preserve captions/logs; no automatic re-decode or retraining.',flush=True)
